<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/wp7-screen-notebook/notebooks/09_fullres_fused_vs_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full resolution: fused transformer against the collapsed control

Two runs at the same settings, one after the other: native 256x256, `FULL_EPOCHS` epochs (60 by default), batch 8, constant learning rate, `--standardize`, one seed. **Runtime → Change runtime type → GPU, then Runtime → Run all.**

| Tag | Model | Flags | Parameters |
|---|---|---|---|
| `tf_fused_full` | BandTransformer 2+6, band tokens | `--model transformer --lr 3e-4` | 488,487 |
| `tf_collapsed_full` | BandTransformer 0+8, bands mixed first (the control) | `--model transformer --tokenizer collapsed --depth-spectral 0 --depth-spatial 8 --lr 3e-4` | 491,191 |
| `cnn_full` | MiniNetSeg | `--model cnn --lr 1e-3` | 571,879 |

`RUN_TAGS` defaults to the fused model, then the control. `cnn_full` is the optional third reference: add it to `RUN_TAGS` to run it too.

| Step | What | Stops when |
|---|---|---|
| Setup | clone `BRANCH`, install, mount Drive | no GPU, the clone fails, or the branch does not take the flags |
| Data | reuse `SW_ROOT`, or copy the zip from Drive, or `scripts/get_data.sh` | md5 or split counts wrong |
| Smoke | two steps of each model at full resolution | a parameter line, the tokenizer line or the resolution is wrong |
| Launch | background `python -m sw.gates run`, one folder per tag | a tag's folder was started with other flags, or a finished tag ran fewer epochs than `FULL_EPOCHS` and `EXTEND` is off |
| Monitor | progress every `POLL_SECONDS` | a run died or is stuck, or its log shows the wrong images, resolution or model |
| Table | best val FG mIoU, per-class IoU, fused minus control | |

Checkpoints, `metrics.jsonl` and `train.log` go to Drive (`RUNS/<tag>/`). The cubes stay on `/content`. If the session dies, open the notebook and Run all again: finished tags are skipped and the others resume from `last.pt`.

**The learning rate is constant on purpose.** A constant-rate run can be trained on to more epochs later: raise `FULL_EPOCHS` and Run all again. See the Settings note.

**Cost (an estimate, not measured for this notebook).** An earlier measurement, not recorded in this repo, gave about 63 s per epoch for the fused model at full resolution, batch 8, on an L4. At that rate 60 epochs is roughly 1 hour per run, plus the data download. The control and the CNN have not been timed at full resolution.

**One seed, `FULL_EPOCHS` epochs (60 by default), validation only.** The protocol in `docs/RECIPE.md` is up to 200 epochs, three seeds, and a warmup-then-cosine schedule. This notebook uses a constant rate, so training it on to 200 epochs is still a screen, not the protocol run. This run does not settle the hypothesis.

AI-assisted notebook; Scott reviews every line.

## Settings

Edit here, or set the environment variables (the local test does that). `FULL_EPOCHS` is set once and passed once, inside `SHARED`. There is no `--downsample` flag: `sw.train` then trains at native size and logs `downsample 1`. Each tag computes its own band statistics into its own folder. No tag takes `--stats-path`.

**Training longer.** With `--no-cosine` the learning rate is the same in every epoch, and `last.pt` holds the weights, the optimizer state and the best score. So a run that finished at 60 epochs can be resumed to 100: the weights, the optimizer, the best score and the metrics log carry on, and the rate does not change. Set `FULL_EPOCHS` higher and Run all again. With `EXTEND = True` the Launch cell reopens every finished tag that has fewer epochs and resumes it. One thing does not carry on: the random stream (shuffle order, flips) starts again from the seed at each resume.

A cosine run cannot be extended this way. Its rate falls towards zero as the planned last epoch nears, and on a resume past that point it climbs back up.

Do not point `FULL_RUNS_DIR` at a folder used for a local test or for other data. A tag that resumes reuses the `band_stats.pt` already in its folder and does not compute it again.

In [1]:
import os
from pathlib import Path

BRANCH = os.environ.get("SW_BRANCH", "wp6-collapsed-tokenizer")  # the branch that has --tokenizer
REPO_URL = "https://github.com/scottn66/255-final-project.git"
RUNS = Path(os.environ.get("FULL_RUNS_DIR", "/content/drive/MyDrive/sw_runs/fullres"))  # Drive: one folder per tag
WORK = Path(os.environ.get("FULL_WORK", "/content/sw_fullres"))  # local disk: the smoke runs
DATA_DIR = Path(os.environ.get("FULL_DATA", "/content/data"))  # local disk: the cubes (never train from Drive)
FULL_ZIP = os.environ.get("FULL_ZIP", "")  # optional: a copy of spectralwaste_segmentation.zip on Drive
DELETE_ZIP = True  # free 21 GB of local disk once the cubes are unpacked and counted
DEVICE = os.environ.get("FULL_DEVICE", "cuda")
NUM_WORKERS = int(os.environ.get("FULL_WORKERS", "2"))
POLL_SECONDS = int(os.environ.get("FULL_POLL", "300"))  # how often the monitor prints
# A live run is called stuck when its train.log and metrics.jsonl have not grown for
# max(STALL_FLOOR_SECONDS, STALL_EPOCHS x its median epoch time). The log grows once per epoch.
# The floor covers the band statistics pass and epoch 1. Their time at full resolution is not measured.
STALL_FLOOR_SECONDS = int(os.environ.get("FULL_STALL_FLOOR", "3600"))
STALL_EPOCHS = 5

FULL_EPOCHS = int(os.environ.get("FULL_EPOCHS", "60"))
# True: a tag that finished with fewer than FULL_EPOCHS epochs is reopened and resumed to FULL_EPOCHS.
# False: the Launch cell stops when it finds such a tag.
EXTEND = os.environ.get("FULL_EXTEND", "1") != "0"

# The same for every tag. --epochs is here and nowhere else. No --downsample: native 256x256.
SHARED = [
    "--batch-size", "8",
    "--no-cosine",  # constant lr, so a run can be extended
    "--standardize",
    "--epochs", str(FULL_EPOCHS),
    "--num-workers", str(NUM_WORKERS),
    "--device", DEVICE,
]

# What differs between the tags.
ARMS = {
    "tf_fused_full": ["--model", "transformer", "--lr", "3e-4"],
    "tf_collapsed_full": [
        "--model", "transformer", "--tokenizer", "collapsed",
        "--depth-spectral", "0", "--depth-spatial", "8", "--lr", "3e-4",
    ],
    "cnn_full": ["--model", "cnn", "--lr", "1e-3"],
}
FULL = {tag: [*flags, *SHARED] for tag, flags in ARMS.items()}
FUSED, CONTROL = "tf_fused_full", "tf_collapsed_full"  # the pair the table compares

# Lines every smoke run and every train.log must show for the tag.
PARAM_LINE = {
    "tf_fused_full": "BandTransformer parameters: 488,487 (0.488M)",
    "tf_collapsed_full": "BandTransformer parameters: 491,191 (0.491M)",
    "cnn_full": "MiniNetSeg parameters: 571,879 (0.572M)",
}
TOKENIZER_LINE = {
    "tf_fused_full": "transformer  tokenizer band  blocks 2+6",
    "tf_collapsed_full": "transformer  tokenizer collapsed  blocks 0+8",
    "cnn_full": None,  # the CNN prints no tokenizer line
}
# What sw.train prints after the image counts when no --downsample flag is given.
RESOLUTION_TEXT = "downsample 1  "

# Which tags this notebook runs, in this order. Add "cnn_full" for the optional CNN reference.
RUN_TAGS = ["tf_fused_full", "tf_collapsed_full"]
if os.environ.get("FULL_TAGS"):
    RUN_TAGS = [tag.strip() for tag in os.environ["FULL_TAGS"].split(",") if tag.strip()]

# True: one run at a time, the next starts when the one before is done.
# GPU memory for two full-resolution runs at once has not been measured. False starts them all together.
ONE_AT_A_TIME = os.environ.get("FULL_ONE_AT_A_TIME", "1") != "0"
WAIT_FOR_RUNS = True  # the monitor waits until every tag is done; interrupting it does not stop a run

unknown = [tag for tag in RUN_TAGS if tag not in FULL]
if unknown or not RUN_TAGS or len(set(RUN_TAGS)) != len(RUN_TAGS):
    raise ValueError(f"RUN_TAGS must be distinct names from {list(FULL)}; got {RUN_TAGS}")
for tag, flags in FULL.items():
    if flags.count("--epochs") != 1:
        raise ValueError(f"{tag}: --epochs must appear once, in SHARED")
    if "--downsample" in flags:
        raise ValueError(f"{tag}: no --downsample here; this notebook trains at native size")
    if "--stats-path" in flags:
        raise ValueError(f"{tag}: no --stats-path here; each tag computes its own band statistics")
    if "--no-cosine" not in flags:
        raise ValueError(f"{tag}: --no-cosine is needed; a cosine run cannot be extended")

## Setup

On Colab: mount Drive, clone `BRANCH` (or pull it if the clone is already there), install `requirements.txt`, check for a GPU, and check that this branch's `sw.train` takes every tag's flags. A branch without `--tokenizer` stops here, before the data step. Locally: use the repo this notebook is in.

After changing `BRANCH`, restart the runtime before Run all. The flag check runs in a new Python process, so it sees the new branch. This notebook's own `sw.gates` import does not: it stays on the code loaded first until the runtime restarts.

In [2]:
import gc
import json
import re
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    """Run a command, show its output, raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


def tail(path, n=25):
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:]) if Path(path).is_file() else "(no log)"


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(
                f"{REPO} has local changes that block switching to {BRANCH}; "
                "delete the folder and Run all again"
            ) from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env) if _env else next(
        p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file()
    )

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch
from sw import gates

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")

COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
ON_BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True
).stdout.strip()

ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}

# Every tag's flags must parse on this branch, before the data step. The check runs in a new
# Python process, as the runs do, so it tests the code on disk and not a module imported earlier.
PARSE_CHECK = "import sys; from sw.train import _build_parser; _build_parser().parse_args(sys.argv[1:])"


def flags_error(flags):
    """None when this branch's sw.train takes the flags, else the last line of its error."""
    check = subprocess.run([sys.executable, "-c", PARSE_CHECK, *flags], capture_output=True, text=True, env=ENV, cwd=REPO)
    if check.returncode == 0:
        return None
    return (check.stderr.strip().splitlines() or [f"exit code {check.returncode}"])[-1]


for tag, flags in FULL.items():
    _why = flags_error(flags)
    if _why:
        raise RuntimeError(
            f"{REPO} (branch {ON_BRANCH or BRANCH}, commit {COMMIT}) does not take the flags of {tag}: {_why}\n"
            "This notebook needs --tokenizer and --standardize. Set BRANCH to wp6-collapsed-tokenizer or a later branch."
        )

RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE

print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"tags   {RUN_TAGS}, {FULL_EPOCHS} epochs each, {'one at a time' if ONE_AT_A_TIME else 'all at once'}, extend {'on' if EXTEND else 'off'}")
print(f"runs   {RUNS}\nwork   {WORK}\ndata   {DATA_DIR}")

Mounted at /content/drive
$ git clone -q -b wp6-collapsed-tokenizer https://github.com/scottn66/255-final-project.git /content/255-final-project
$ /usr/bin/python3 -m pip install -q -r /content/255-final-project/requirements.txt
repo   /content/255-final-project @ 318358e (branch wp6-collapsed-tokenizer)
torch  2.14.1+cu130 on NVIDIA L4
tags   ['tf_fused_full', 'tf_collapsed_full'], 60 epochs each, one at a time, extend on
runs   /content/drive/MyDrive/sw_runs/fullres
work   /content/sw_fullres
data   /content/data


## Data

Three ways, tried in this order:

1. `SW_ROOT` is set and already holds the full train and val splits. Nothing is downloaded.
2. `FULL_ZIP` points at a copy of the zip on Drive. It is copied to local disk, then checked and unpacked.
3. `scripts/get_data.sh` downloads the zip (resumable), checks its md5, and unzips it.

The counts must be 514 train and 167 val. The monitor checks the same two numbers, and `downsample 1`, in every run's log.

In [3]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def split_sizes(root):
    return {s: len(SpectralWasteDataset(root, split=s, random_flip=False)) for s in ("train", "val")}


def counts_ok(root):
    try:
        return all(n == SPLIT_COUNTS[s] for s, n in split_sizes(root).items())
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("FULL_ALLOW_PARTIAL") == "1"  # local test only: a small SW_ROOT
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test; not the official split)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if FULL_ZIP and Path(FULL_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", FULL_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)

# What every train.log must show: 514 and 167 (unless this is the local test), at native size.
SIZES = split_sizes(SW_ROOT)
IMAGES_LINE = f"train images {SIZES['train']}  val images {SIZES['val']}  {RESOLUTION_TEXT}"
print(f"every run must log: {IMAGES_LINE.strip()}")

local disk free: 235 GiB (zip 21 GB + cubes 24 GB while unzipping)
$ bash /content/255-final-project/scripts/get_data.sh /content/data
every run must log: train images 514  val images 167  downsample 1


## Smoke: one per model, at full resolution

Two optimizer steps of each of the three models on 4 training images, with `--standardize`, before any long run. `--smoke` alone would shrink the images by 4, so this cell passes `--downsample 1`: the smoke takes the same full-resolution path as the real runs. (The real runs get there by having no `--downsample` flag.)

Each smoke must print its parameter line and `downsample 1`. The fused model must read `488,487`, the control `491,191` and `transformer  tokenizer collapsed  blocks 0+8`. A wrong line stops the notebook: it would mean this branch does not build the model the table names.

On a GPU the cell then prints the peak GPU memory of one forward and backward pass on a random batch of 8 at 224 x 256 x 256, for each tag in `RUN_TAGS`. It leaves out the optimizer step and the data loader, so a real step can use a little more. A failure there (out of memory, for example) is printed and does not stop the notebook; the memory is handed back before the next tag and before the Launch cell. On CPU that part is skipped. **This memory part has not been run on a GPU yet.**

In [4]:
for tag, flags in ARMS.items():
    smoke = subprocess.run(
        [
            sys.executable, "-m", "sw.train", *flags, "--smoke", "--downsample", "1", "--standardize",
            "--device", DEVICE, "--data-root", str(SW_ROOT),
            "--out-dir", str(WORK / f"smoke_{tag}"),
        ],
        capture_output=True, text=True, env=ENV, cwd=REPO,
    )
    print(f"--- smoke {tag}")
    print(smoke.stdout[-1500:], smoke.stderr[-3000:], sep="\n")
    if smoke.returncode != 0:
        raise RuntimeError(f"smoke {tag} failed: the run raised (traceback above)")
    for line in (PARAM_LINE[tag], TOKENIZER_LINE[tag], "  " + RESOLUTION_TEXT + "crop None"):
        if line and line not in smoke.stdout:
            raise RuntimeError(f"smoke {tag} failed: the output has no line '{line}'")
print("smoke passed for", ", ".join(ARMS))


def one_batch_peak_gib(tag, size=256, bands=224):
    """One forward and backward pass of the tag's model on a random batch of its batch size.

    Returns the peak GPU memory in GiB, or None when DEVICE is not cuda.
    """
    from sw.train import _build_parser, build_model, config_from_args

    args = _build_parser().parse_args(FULL[tag])
    cfg = config_from_args(args)
    on_gpu = DEVICE == "cuda"
    model = build_model(cfg, in_channels=bands).to(DEVICE)
    if on_gpu:
        torch.cuda.reset_peak_memory_stats()
    cubes = torch.randn(cfg.batch_size, bands, size, size, device=DEVICE)
    masks = torch.randint(0, 7, (cfg.batch_size, size, size), device=DEVICE)
    loss = torch.nn.functional.cross_entropy(model(cubes), masks)
    loss.backward()
    return torch.cuda.max_memory_allocated() / 2**30 if on_gpu else None


def measure_memory(tags, measure=one_batch_peak_gib):
    """One printed line per tag. A failure is printed, not raised: the real run's own log is the check that counts."""
    for tag in tags:
        try:
            line = f"{measure(tag):.2f} GiB"
        except Exception as err:
            line = f"could not be measured: {type(err).__name__}: {str(err)[:200]}"
        # Free the memory here, after the except block has ended. Inside it the error still
        # holds the model and the batch, so the cache could not be emptied.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  {tag:18s} {line}")


if DEVICE == "cuda":
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"\npeak GPU memory, one forward and backward pass, batch 8 at 224 x 256 x 256 (GPU has {total:.1f} GiB)")
    measure_memory(RUN_TAGS)  # only the tags that will run
    print(f"  held by this notebook after the check: {torch.cuda.memory_reserved() / 2**30:.2f} GiB")
else:
    print(f"\npeak GPU memory: skipped (device is {DEVICE})")

--- smoke tf_fused_full
standardize: train band stats -> /content/sw_fullres/smoke_tf_fused_full/band_stats.pt  (C=224)
BandTransformer parameters: 488,487 (0.488M)
transformer  tokenizer band  blocks 2+6  dim 64  heads 4  mlp_ratio 4.0  patch 8
device cuda  epochs 1-1  batch 2  lr 0.0003  weight_decay 0.0001  schedule cosine  amp off
train images 4  val images 2  downsample 1  crop None  augment on  out /content/sw_fullres/smoke_tf_fused_full
class weights (paper): 0.676 0.937 1.000 1.207 1.125 1.516 0.900
Limited run: the val FG mIoU below checks the wiring. It is not a score to report.
epoch 01  steps    2  lr 0.0003  train_loss 1.9954  train_fg_miou 0.0750  val_fg_miou 0.0849  3.3s
saved /content/sw_fullres/smoke_tf_fused_full/best.pt  (epoch 1, val FG mIoU 0.0849)
saved /content/sw_fullres/smoke_tf_fused_full/last.pt  (epoch 1)
done  best val FG mIoU 0.0849  epoch 1  log /content/sw_fullres/smoke_tf_fused_full/metrics.jsonl


--- smoke tf_collapsed_full
standardize: train band sta

## Launch

Each tag is a background `python -m sw.gates run` with its checkpoints and log in `RUNS/<tag>/`. Interrupting a cell does not stop it. Re-running this cell skips finished tags, leaves a live run alone, and resumes a stopped one from its `last.pt`.

With `ONE_AT_A_TIME` this cell starts the first unfinished tag only. The monitor starts the next one when it is done, so keep the monitor running.

First the cell compares the flags stored in every tag's folder (`ARGS.json`) with this notebook's flags for that tag. `--epochs`, `--data-root`, `--device` and `--num-workers` may differ; anything else (batch size, learning rate, schedule, `--standardize`, the model) stops the cell before any file is changed. This covers finished tags too.

A finished tag is skipped when it ran `FULL_EPOCHS` epochs or more. When it ran fewer:

* `EXTEND = True`: the cell moves its `DONE.json` aside (to `DONE.after_<n>_epochs.json`), prints a line saying so, and the run resumes from `last.pt` up to `FULL_EPOCHS`.
* `EXTEND = False`: the cell stops and names the tag.

A run that is still training keeps the `--epochs` it was started with. If `FULL_EPOCHS` was raised since, the monitor reopens the run when it finishes and trains it on to `FULL_EPOCHS` (with `EXTEND = False` the monitor stops instead).

In [5]:
PROCS = {}
STARTED = set()  # tags started, or seen running, since this cell last ran
DEAD = set()  # tags the monitor saw die; re-running this cell clears it


def epochs_done(tag):
    """The last epoch in the tag's metrics.jsonl, or None."""
    try:
        return int(gates.read_metrics(RUNS / tag / "metrics.jsonl")[-1]["epoch"])
    except (FileNotFoundError, ValueError):
        return None


def train_flags(tag):
    """The sw.train flags the tag is launched with."""
    return [*FULL[tag], "--data-root", str(SW_ROOT)]


def flags_differ(tag):
    """None when the tag's folder has no ARGS.json or was started with this notebook's flags.

    Otherwise (stored, now): the two flag lists without the flags a resume may change.
    """
    stored = RUNS / tag / gates.ARGS_FILE
    if not stored.is_file():
        return None
    before = gates.run_identity(json.loads(stored.read_text(encoding="utf-8")))
    now = gates.run_identity(train_flags(tag))
    return None if before == now else (before, now)


def check_flags():
    """Raise if a tag's folder, finished or not, was started with other flags. Changes nothing."""
    for tag in RUN_TAGS:
        differ = flags_differ(tag)
        if differ:
            raise RuntimeError(
                f"{tag}: {RUNS / tag} was started with different flags.\n  stored: {differ[0]}\n  now:    {differ[1]}\n"
                "Nothing was changed. Its numbers are not from this notebook's settings. "
                "Put the settings back, or set another runs folder (FULL_RUNS_DIR), or move this folder away."
            )


def short_tags():
    """Finished tags that ran fewer epochs than FULL_EPOCHS, as {tag: epochs run}."""
    return {
        tag: epochs_done(tag) or 0 for tag in RUN_TAGS
        if gates.run_state(RUNS / tag) == "done" and (epochs_done(tag) or 0) < FULL_EPOCHS
    }


def extend_or_refuse():
    """Reopen finished tags that are short of FULL_EPOCHS (EXTEND on), or raise (EXTEND off)."""
    short = short_tags()
    if not short:
        return
    if not EXTEND:
        raise RuntimeError(
            f"finished with fewer than {FULL_EPOCHS} epochs: {short} (tag: epochs run) in {RUNS}.\n"
            f"Set EXTEND = True to train such a tag on to {FULL_EPOCHS}, or lower FULL_EPOCHS, or set another runs folder."
        )
    no_checkpoint = [tag for tag in short if not (RUNS / tag / "last.pt").is_file()]
    if no_checkpoint:
        raise RuntimeError(
            f"{no_checkpoint}: finished with fewer than {FULL_EPOCHS} epochs and there is no last.pt to resume from "
            f"in {RUNS}. Nothing was changed. Put last.pt back, or move the folder away to train from epoch 1."
        )
    for tag, n in short.items():
        done = RUNS / tag / gates.DONE_FILE
        kept = done.with_name(f"DONE.after_{n}_epochs.json")
        done.replace(kept)
        print(
            f"EXTEND {tag}: it finished at {n} epochs and FULL_EPOCHS is {FULL_EPOCHS}. "
            f"Moved {gates.DONE_FILE} to {kept.name}; the run resumes from last.pt at epoch {n + 1}."
        )


def start_due():
    """Start the tags that should be training now. A tag that died is not started again here."""
    for tag in RUN_TAGS:
        if tag in DEAD:
            continue
        state = gates.run_state(RUNS / tag)
        if state == "done":
            continue
        if state != "running" and tag not in STARTED:
            PROCS[tag] = gates.launch(RUNS / tag, train_flags(tag), env=ENV)
        STARTED.add(tag)
        if ONE_AT_A_TIME:
            return


check_flags()
extend_or_refuse()
start_due()

tf_fused_full: started pid 9854 (new); log /content/drive/MyDrive/sw_runs/fullres/tf_fused_full/train.log


## Monitor

Prints every tag's progress every `POLL_SECONDS`. Interrupt it any time: the runs keep going, and re-running it picks them up.

It stops with an error in three cases.

* **A run died.** Its folder has no `DONE.json` and its process is gone. The end of its `train.log` is printed. The other tags still run to the end; then this cell raises and names the dead ones. Fix the cause, then re-run the Launch cell and this cell: the run resumes from `last.pt`.
* **A run is stuck.** Its process is alive, but its `train.log` and `metrics.jsonl` have not grown for `max(STALL_FLOOR_SECONDS, STALL_EPOCHS x median epoch time)`. The cell raises at once and prints the pid. Nothing is killed. The clock starts when this cell starts.
* **A log shows the wrong run.** Every `train.log` must show the image counts from the Data cell (514 and 167 on the real set) followed by `downsample 1`, the tag's parameter count, and for the transformers the tokenizer line.

In [6]:
try:
    from tqdm.auto import tqdm
except ImportError:  # no progress bars; the printed lines are enough
    tqdm = None


def log_check(tag):
    """Compare the banner lines in the tag's train.log with what the tag must show.

    Returns (complete, wrong). complete: every expected line was found at
    least once. wrong: banner lines that do not match. A resumed run prints
    the banner again, so every copy is checked.
    """
    log = RUNS / tag / "train.log"
    lines = log.read_text(errors="replace").splitlines() if log.is_file() else []
    images = [line for line in lines if line.startswith("train images ")]
    params = [line for line in lines if re.match(r"\w+ parameters: ", line)]
    tokenizer = [line for line in lines if line.startswith("transformer  tokenizer ")]
    wrong = [line for line in images if not line.startswith(IMAGES_LINE)]
    wrong += [line for line in params if line != PARAM_LINE[tag]]
    want = TOKENIZER_LINE[tag]
    wrong += [line for line in tokenizer if want is None or not line.startswith(want + "  ")]
    complete = bool(images) and bool(params) and (want is None or bool(tokenizer))
    return complete, wrong


def status(tag):
    out = RUNS / tag
    state = gates.run_state(out)
    proc = PROCS.get(tag)
    if proc is not None and proc.poll() is None:
        state = "running"
    if state == "new" and tag not in STARTED:
        return state, f"{tag:18s} waiting  not started yet"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return state, f"{tag:18s} {state:8s} no epoch finished yet"
    last = rows[-1]
    best = max(rows, key=lambda r: r.get("val_fg_miou") or -1.0)
    sec = gates.median_epoch_seconds(rows) or 0.0
    left = sec * max(0, FULL_EPOCHS - int(last["epoch"])) / 3600 if state != "done" else 0.0
    return state, (
        f"{tag:18s} {state:8s} epoch {last['epoch']:3d}/{FULL_EPOCHS}  "
        f"val FG {last.get('val_fg_miou') or 0:.3f} (best {best.get('val_fg_miou') or 0:.3f} at {best['epoch']})  "
        f"{sec:.1f} s/epoch  about {left:.1f} h left"
    )


GROWTH = {}  # tag -> (log sizes, when this cell first saw them)


def quiet_seconds(tag):
    """Seconds since this cell last saw the tag's train.log or metrics.jsonl grow."""
    out = RUNS / tag
    sizes = tuple(
        (out / name).stat().st_size if (out / name).is_file() else -1 for name in ("train.log", "metrics.jsonl")
    )
    if tag not in GROWTH or GROWTH[tag][0] != sizes:
        GROWTH[tag] = (sizes, time.time())
    return time.time() - GROWTH[tag][1]


def stall_limit(tag):
    try:
        sec = gates.median_epoch_seconds(gates.read_metrics(RUNS / tag / "metrics.jsonl")) or 0.0
    except (FileNotFoundError, ValueError):
        sec = 0.0
    return max(STALL_FLOOR_SECONDS, STALL_EPOCHS * sec)


BARS = {} if tqdm is None else {
    tag: tqdm(total=FULL_EPOCHS, desc=tag, unit="epoch", position=i) for i, tag in enumerate(RUN_TAGS)
}


def update_bar(tag):
    if tag not in BARS:
        return
    try:
        rows = gates.read_metrics(RUNS / tag / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return
    best = max(r.get("val_fg_miou") or 0.0 for r in rows)
    BARS[tag].n = int(rows[-1]["epoch"])  # set, not add: safe after a resume
    BARS[tag].set_postfix(best_val=f"{best:.3f}")
    BARS[tag].refresh()


while True:
    if EXTEND and short_tags():
        # A run started at an older, lower FULL_EPOCHS has finished. Reopen it and train it on.
        STARTED.difference_update(short_tags())
        extend_or_refuse()
        start_due()
    report = {tag: status(tag) for tag in RUN_TAGS}
    print(time.strftime("%H:%M:%S"), *[line for _, line in report.values()], sep="\n  ", flush=True)
    for tag in RUN_TAGS:
        update_bar(tag)
        _, wrong = log_check(tag)
        if wrong:
            shown = "\n  ".join(wrong)
            raise RuntimeError(
                f"{tag}: {RUNS / tag / 'train.log'} shows the wrong run:\n  {shown}\n"
                f"expected '{IMAGES_LINE.strip()}', '{PARAM_LINE[tag]}'"
                + (f", '{TOKENIZER_LINE[tag]}'" if TOKENIZER_LINE[tag] else "")
                + f".\nIf it is still training, stop it (its pid is in {RUNS / tag / gates.RUNNING_FILE}). "
                "Then move the folder away and Run all again."
            )
    states = {tag: state for tag, (state, _) in report.items()}
    for tag in RUN_TAGS:
        if tag in STARTED and tag not in DEAD and states[tag] in ("new", "partial"):
            DEAD.add(tag)
            print(f"\n{tag} is not running and not done. End of its log:\n{tail(RUNS / tag / 'train.log')}\n", flush=True)
    for tag in RUN_TAGS:
        if states[tag] == "running":
            quiet, limit = quiet_seconds(tag), stall_limit(tag)
            if quiet > limit:
                print(f"\n{tag} is alive but its log has not grown. End of its log:\n{tail(RUNS / tag / 'train.log')}")
                raise RuntimeError(
                    f"{tag} is stuck: no new line in train.log or metrics.jsonl for {quiet:.0f} s (limit {limit:.0f} s). "
                    f"Its pid is in {RUNS / tag / gates.RUNNING_FILE}. Stop it, then re-run the Launch cell and this cell"
                )
    if all(states[tag] == "done" or tag in DEAD for tag in RUN_TAGS):
        break
    if not WAIT_FOR_RUNS:
        break
    start_due()
    time.sleep(POLL_SECONDS)

if DEAD:
    raise RuntimeError(
        f"{sorted(DEAD)} died (log tails above). Fix the cause, then re-run the Launch cell and this cell; "
        "a run with a last.pt resumes from it"
    )
if short_tags():
    raise RuntimeError(
        f"finished with fewer than {FULL_EPOCHS} epochs: {short_tags()} (tag: epochs run) and EXTEND is off. "
        f"Set EXTEND = True to train such a tag on to {FULL_EPOCHS}, or lower FULL_EPOCHS; then re-run the Launch cell and this cell."
    )

tf_fused_full:   0%|          | 0/60 [00:00<?, ?epoch/s]

tf_collapsed_full:   0%|          | 0/60 [00:00<?, ?epoch/s]

23:10:02
  tf_fused_full      running  no epoch finished yet
  tf_collapsed_full  waiting  not started yet
23:15:02
  tf_fused_full      running  epoch   3/60  val FG 0.000 (best 0.000 at 1)  65.4 s/epoch  about 1.0 h left
  tf_collapsed_full  waiting  not started yet
23:20:02
  tf_fused_full      running  epoch   7/60  val FG 0.000 (best 0.000 at 1)  63.7 s/epoch  about 0.9 h left
  tf_collapsed_full  waiting  not started yet
23:25:02
  tf_fused_full      running  epoch  12/60  val FG 0.000 (best 0.000 at 1)  63.9 s/epoch  about 0.9 h left
  tf_collapsed_full  waiting  not started yet
23:30:02
  tf_fused_full      running  epoch  17/60  val FG 0.096 (best 0.096 at 17)  63.7 s/epoch  about 0.8 h left
  tf_collapsed_full  waiting  not started yet
23:35:02
  tf_fused_full      running  epoch  21/60  val FG 0.096 (best 0.096 at 21)  63.6 s/epoch  about 0.7 h left
  tf_collapsed_full  waiting  not started yet
23:40:02
  tf_fused_full      running  epoch  26/60  val FG 0.250 (best 0.257 at 

## Table

One row per tag: best val FG mIoU and its epoch, the epochs run, the mean val FG mIoU of the last 10 epochs, the median seconds per epoch (epoch 1 left out), and the parameter count. `log` is `ok` when the run's `train.log` shows the expected images, resolution and model. `flags` is `ok` when the flags stored in the run's folder are this notebook's flags for the tag (see Launch), and `DIFFER` when they are not: such a row is not from these settings. Then the per-class val IoU at the best epoch.

**Still rising?** For each tag: the mean val FG mIoU of the last 10 epochs against the 10 before. More than 1 point up is printed as `still rising`. That is a rule of thumb to help decide whether to raise `FULL_EPOCHS` and extend; it is not a test.

**Fused minus control**, in points (1 point = 0.01 IoU), two ways: the best epoch of each run, and the mean of the last 10 epochs of each run. The single best epoch is noisy. In the half-resolution screen the CNN's best epoch was 0.403 and its last epoch was 0.277. The two runs must have the same number of epochs. If they do not, a warning line is printed and the comparison uses only the epochs both runs have.

In [7]:
from sw.data import CLASS_NAMES

LAST = 10  # epochs in the "last epochs" mean
RISING_POINTS = 1.0  # last 10 against the 10 before: more than this is printed as "still rising"


def cell(value):
    return "  n/a" if value is None else f"{value:.3f}"


def mean(values):
    kept = [v for v in values if v is not None]
    return sum(kept) / len(kept) if kept else None


def points(a, b):
    """a minus b in points (1 point = 0.01 IoU), as text."""
    return "   n/a" if a is None or b is None else f"{100 * (a - b):+6.1f}"


def summarize(rows):
    """Best epoch, and means over the last LAST epochs, of a list of metrics rows."""
    best = max(rows, key=lambda r: -1.0 if r.get("val_fg_miou") is None else r["val_fg_miou"])
    late = rows[-LAST:]
    before = rows[-2 * LAST:-LAST] if len(rows) >= 2 * LAST else []
    return {
        "best_val_fg_miou": best.get("val_fg_miou"),
        "best_epoch": best["epoch"],
        "epochs": rows[-1]["epoch"],
        "per_class_iou_at_best": best.get("per_class_iou") or {},
        "last_epochs": len(late),
        "last_mean": mean(r.get("val_fg_miou") for r in late),
        "before_mean": mean(r.get("val_fg_miou") for r in before),
        "per_class_last_mean": {
            name: mean((r.get("per_class_iou") or {}).get(name) for r in late) for name in CLASS_NAMES
        },
    }


print(
    f"{'tag':18s} {'state':8s} {'best val FG':>11s} {'epoch':>5s} {'epochs':>6s} {'last-10 mean':>12s} "
    f"{'s/epoch':>8s} {'parameters':>10s}  {'log':9s} flags"
)
ROWS, TABLE = {}, {}
for tag in RUN_TAGS:
    out = RUNS / tag
    state = gates.run_state(out)
    complete, wrong = log_check(tag)
    log = "WRONG" if wrong else "ok" if complete else "not found"
    flags = "DIFFER" if flags_differ(tag) else "ok" if (out / gates.ARGS_FILE).is_file() else "no record"
    try:
        ROWS[tag] = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        print(f"{tag:18s} {state:8s} no epoch finished yet")
        continue
    params = None
    if (out / gates.DONE_FILE).is_file():
        params = json.loads((out / gates.DONE_FILE).read_text())["result"].get("param_count")
    TABLE[tag] = row = {
        **summarize(ROWS[tag]),
        "state": state,
        "median_epoch_seconds": gates.median_epoch_seconds(ROWS[tag]),
        "param_count": params,
        "log": log,
        "flags": flags,
    }
    sec = row["median_epoch_seconds"]
    print(
        f"{tag:18s} {state:8s} {cell(row['best_val_fg_miou']):>11s} {row['best_epoch']:5d} {row['epochs']:6d} "
        f"{cell(row['last_mean']):>12s} {'n/a' if sec is None else format(sec, '.1f'):>8s} "
        f"{'n/a' if params is None else format(params, ','):>10s}  {log:9s} {flags}"
    )

print("\nper-class val IoU at the best epoch")
print(f"{'tag':18s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
for tag, row in TABLE.items():
    print(f"{tag:18s} " + " ".join(f"{cell(row['per_class_iou_at_best'].get(name)):>10s}" for name in CLASS_NAMES))

print(f"\nstill rising? mean val FG mIoU of the last {LAST} epochs against the {LAST} before")
for tag, row in TABLE.items():
    if row["before_mean"] is None or row["last_mean"] is None:
        print(f"{tag:18s} n/a (needs {2 * LAST} epochs; has {row['epochs']})")
        continue
    gain = 100 * (row["last_mean"] - row["before_mean"])
    word = "still rising" if gain > RISING_POINTS else "not clearly rising"
    print(
        f"{tag:18s} {row['before_mean']:.3f} -> {row['last_mean']:.3f}  {gain:+.1f} points  {word} "
        f"(rule of thumb: more than {RISING_POINTS:.0f} point up)"
    )

unfinished = [tag for tag in RUN_TAGS if gates.run_state(RUNS / tag) != "done"]
if unfinished:
    print(f"\nnot finished: {unfinished}. Their rows are not final; re-run this cell when they are done.")

differ = [tag for tag, row in TABLE.items() if row["flags"] == "DIFFER"]
if differ:
    print(f"\nWARNING: {differ} were started with flags that differ from this notebook's. Their rows are not from these settings.")

print("\nfused minus control, in points (1 point = 0.01 IoU)")
if FUSED in ROWS and CONTROL in ROWS:
    n_fused, n_control = TABLE[FUSED]["epochs"], TABLE[CONTROL]["epochs"]
    common = min(n_fused, n_control)
    print(f"epochs run: {FUSED} {n_fused}, {CONTROL} {n_control}")
    if n_fused != n_control:
        print(
            f"WARNING: the two runs have different epoch counts ({n_fused} and {n_control}). "
            f"The comparison below uses epochs 1 to {common} of each. "
            "Raise FULL_EPOCHS to the larger count and re-run the Launch cell to bring the shorter run level."
        )
    fused = summarize([r for r in ROWS[FUSED] if r["epoch"] <= common])
    control = summarize([r for r in ROWS[CONTROL] if r["epoch"] <= common])
    COMPARISON = {"epochs": common, "fused": fused, "control": control}
    print(f"{'':44s} {'FG mIoU':>7s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
    label = f"best epoch of each (fused {fused['best_epoch']}, control {control['best_epoch']})"
    print(
        f"{label:44s} {points(fused['best_val_fg_miou'], control['best_val_fg_miou']):>7s} "
        + " ".join(
            f"{points(fused['per_class_iou_at_best'].get(name), control['per_class_iou_at_best'].get(name)):>10s}"
            for name in CLASS_NAMES
        )
    )
    label = f"mean of the last {fused['last_epochs']} epochs of each"
    print(
        f"{label:44s} {points(fused['last_mean'], control['last_mean']):>7s} "
        + " ".join(
            f"{points(fused['per_class_last_mean'][name], control['per_class_last_mean'][name]):>10s}"
            for name in CLASS_NAMES
        )
    )
    print("The single best epoch is noisy; read the last-epochs row with it.")
else:
    print(f"needs both {FUSED} and {CONTROL}; not compared.")

print("\nFor reference, the half-resolution screen (notebook 08, commit 318358e, 30 epochs, batch 4). Not these runs.")
print(f"{'tag':18s} {'best val FG':>11s} {'epoch':>5s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
for name, best, epoch, ious in (
    ("tf_fused_std", 0.286, 30, (0.847, 0.437, 0.367, 0.622, 0.000, 0.000, 0.288)),
    ("tf_collapsed_std", 0.298, 28, (0.837, 0.465, 0.375, 0.618, 0.000, 0.000, 0.327)),
    ("cnn_std", 0.403, 25, (0.860, 0.543, 0.517, 0.599, 0.302, 0.000, 0.458)),
):
    print(f"{name:18s} {best:11.3f} {epoch:5d} " + " ".join(f"{v:10.3f}" for v in ious))

ran = ", ".join(f"{tag} {row['epochs']}" for tag, row in TABLE.items()) or "none"
print(
    f"\nOne seed, validation only. Epochs run (of {FULL_EPOCHS} asked): {ran}. "
    "The protocol in docs/RECIPE.md is up to 200 epochs, three seeds, and a warmup-then-cosine schedule. "
    "This notebook uses a constant rate, so training it on to 200 epochs is still a screen, not the protocol run. "
    "This run does not settle the hypothesis. Report the difference in points as printed. "
    'Do not use a stronger word than "competitive" for any comparison, and only with the numbers beside it.'
)
print(f"commit {COMMIT}  runs {RUNS}")
print(f"notebook ran in {(time.time() - T_START) / 60:.1f} min")

tag                state    best val FG epoch epochs last-10 mean  s/epoch parameters  log       flags
tf_fused_full      done           0.321    60     60        0.299     63.4    488,487  ok        ok
tf_collapsed_full  done           0.332    59     60        0.320     63.3    491,191  ok        ok

per-class val IoU at the best epoch
tag                background       film     basket  cardboard video_tape   filament        bag
tf_fused_full           0.858      0.464      0.402      0.692      0.005      0.000      0.364
tf_collapsed_full       0.854      0.507      0.427      0.695      0.023      0.000      0.343

still rising? mean val FG mIoU of the last 10 epochs against the 10 before
tf_fused_full      0.284 -> 0.299  +1.6 points  still rising (rule of thumb: more than 1 point up)
tf_collapsed_full  0.299 -> 0.320  +2.1 points  still rising (rule of thumb: more than 1 point up)

fused minus control, in points (1 point = 0.01 IoU)
epochs run: tf_fused_full 60, tf_collapsed_fu

## What to send back

* The table above, and `RUNS/<tag>/metrics.jsonl` and `RUNS/<tag>/DONE.json` for each tag.
* `docs/RESULTS.md` gets one row per tag, with its flags, the epochs run, and the words *one seed, validation only* in the notes.

To train longer, raise `FULL_EPOCHS` and Run all again. Both tags are then resumed to the new count, one after the other. That is a longer screen. It is not the protocol run: the protocol in `docs/RECIPE.md` is up to 200 epochs, three seeds, and a warmup-then-cosine schedule, and this notebook uses a constant rate.

Nothing here touches the test split.